# Trees and Ensembles, Notebook 3
## Overfitting and pruning: a tree that memorizes the data is useless

**Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### What this is about

A tree can keep splitting until every leaf holds a single firm. Then it gets
100% of the training data right, and it has learned the noise: the details of *these* 146 firms that do not
recur in the other 73. It is the high-degree polynomial, in tree form.

Today we measure that phenomenon carefully and look at the three knobs that control it: not letting it
grow (**pre-pruning**), letting it grow and then cutting it back (**cost-complexity post-pruning**),
and the tool for deciding how much: **cross-validation**.

### What you will learn today

1. What overfitting looks like in a tree: the training and test accuracy curve.
2. Pre-pruning: maximum depth, minimum firms per leaf.
3. Post-pruning: Breiman's cost-complexity and its parameter $\alpha$.
4. Cross-validation: choosing the complexity without spending the test data.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
print("Ready.")


## 1. The data, once more

The same 219 firms and the same partition as in Notebook 2.


| Ratio | What it measures |
|---|---|
| `deuda_activos` | total debt / total assets (leverage) |
| `razon_corriente` | current assets / current liabilities (liquidity) |
| `ventas_deuda` | sales / total debt (ability to service debt) |
| `ln_activos` | logarithm of total assets (size) |
| `roa` | net income / total assets (profitability) |
| `impago` | **what we want to predict:** 1 if the firm defaulted, 0 otherwise |


In [ ]:

# The 219 firms with financial ratios (the default table from the course, with 5 of its ratios).
# It is pasted right here so the notebook does not depend on any external file.
from io import StringIO
IMPAGO_CSV = """deuda_activos,razon_corriente,ventas_deuda,ln_activos,roa,impago
0.374,2.337,1.436,14.697,0.073,0.0
0.435,2.638,1.385,14.877,0.042,0.0
0.443,2.099,0.452,14.931,0.016,0.0
0.23,2.506,4.217,15.586,-0.041,0.0
0.317,2.273,3.116,15.708,0.021,0.0
0.312,3.282,4.842,15.785,0.047,0.0
0.628,1.386,2.91,14.066,0.064,0.0
0.64,1.777,2.895,14.236,0.068,0.0
0.719,1.44,1.505,14.697,0.075,0.0
0.551,1.078,1.944,15.564,0.113,0.0
0.541,0.991,2.091,15.581,0.022,0.0
0.575,0.995,1.787,15.738,0.035,0.0
0.454,3.266,7.321,14.877,0.064,0.0
0.574,3.017,3.159,14.457,0.07,0.0
0.526,3.333,1.962,14.399,0.077,0.0
0.724,0.8,1.625,14.036,0.016,0.0
0.508,1.406,4.356,14.349,0.406,0.0
0.494,1.051,2.299,14.515,0.117,0.0
0.572,1.557,2.106,14.016,0.126,0.0
0.578,1.498,2.033,14.293,0.086,0.0
0.558,1.605,2.035,14.52,0.099,0.0
0.314,2.127,10.633,13.738,0.087,0.0
0.467,1.661,5.599,14.219,0.094,0.0
0.499,1.482,3.109,14.424,0.067,0.0
0.259,3.734,5.692,14.172,0.234,0.0
0.202,4.804,7.312,14.417,0.195,0.0
0.268,3.663,6.08,14.897,0.228,0.0
0.409,1.671,4.523,14.253,0.12,0.0
0.307,2.171,6.817,14.38,0.172,0.0
0.258,3.481,10.557,14.454,0.274,0.0
0.393,1.55,3.81,14.68,0.024,0.0
0.558,1.102,2.597,14.108,0.005,0.0
0.432,1.365,3.534,14.01,0.119,0.0
0.53,1.281,3.099,12.687,0.272,0.0
0.244,3.411,16.602,13.143,0.489,0.0
0.354,2.877,3.112,13.311,0.075,0.0
0.381,1.56,2.749,14.696,0.074,0.0
0.411,2.198,2.46,14.839,0.043,0.0
0.401,1.954,2.067,14.915,0.037,0.0
0.387,1.955,2.616,14.835,0.342,0.0
0.438,1.029,2.619,14.929,0.023,0.0
0.413,1.44,2.869,14.889,-0.011,0.0
0.891,1.118,2.023,13.415,0.036,0.0
1.052,0.741,1.8,14.355,-0.093,0.0
0.466,0.899,1.509,14.364,0.034,0.0
0.491,1.196,2.335,14.414,-0.017,0.0
0.505,1.053,2.678,14.631,0.076,0.0
0.354,2.45,3.08,14.453,0.084,0.0
0.464,1.843,2.521,14.559,0.119,0.0
0.566,1.286,1.675,14.838,0.111,0.0
0.543,1.766,2.619,14.933,0.029,0.0
0.855,1.131,1.805,15.005,0.029,0.0
0.558,1.744,2.516,15.074,0.028,0.0
0.736,1.153,1.564,14.292,0.054,0.0
0.708,1.192,1.352,14.432,0.034,0.0
0.657,1.269,1.395,14.387,0.027,0.0
0.653,1.061,1.588,14.303,0.04,0.0
0.619,1.005,2.113,14.235,0.038,0.0
0.197,9.632,5.15,14.586,0.032,0.0
0.284,1.461,6.585,13.479,0.11,0.0
0.3,1.438,5.251,13.541,0.012,0.0
0.283,2.928,5.778,13.561,0.016,0.0
0.476,1.788,5.329,13.143,0.123,0.0
0.47,1.759,5.033,13.329,0.13,0.0
0.459,2.104,4.49,13.372,0.076,0.0
0.388,2.409,5.319,13.283,0.201,0.0
0.547,1.979,2.09,13.806,0.15,0.0
0.568,1.812,1.937,13.914,0.071,0.0
0.461,1.923,1.499,14.059,-0.013,0.0
0.538,2.174,1.801,14.208,0.025,0.0
0.497,1.878,1.917,14.284,0.075,0.0
0.208,3.306,5.181,15.336,-0.003,0.0
0.241,3.005,4.709,15.428,0.011,0.0
0.222,3.11,3.776,15.427,0.021,0.0
0.392,2.268,6.865,12.39,0.334,0.0
0.252,5.229,7.751,12.318,0.176,0.0
0.849,1.56,1.326,12.304,-0.186,0.0
0.36,2.671,5.196,12.142,0.186,0.0
0.028,34.514,71.252,12.295,0.292,0.0
0.631,1.369,1.835,13.284,0.136,0.0
0.507,1.464,1.625,13.732,-0.217,0.0
0.578,1.577,2.553,13.963,0.02,0.0
0.448,1.681,0.566,13.768,0.024,0.0
0.363,2.721,2.762,13.301,0.446,0.0
0.403,2.199,2.673,13.506,0.067,0.0
0.4,2.237,1.139,13.576,0.043,0.0
0.432,1.579,2.39,14.7,0.066,0.0
0.464,1.763,2.067,14.843,0.059,0.0
0.475,1.707,1.833,14.95,0.06,0.0
0.089,8.849,16.344,15.39,0.013,0.0
0.125,7.175,11.171,15.513,0.059,0.0
0.125,7.156,12.795,15.55,0.041,0.0
0.539,1.764,3.733,15.384,0.088,0.0
0.502,1.79,3.797,15.397,0.035,0.0
0.522,1.572,2.624,15.391,0.029,0.0
0.77,0.383,1.984,15.834,-0.046,0.0
0.746,0.465,3.31,16.041,0.057,0.0
0.666,0.539,3.919,16.051,0.018,0.0
0.14,8.346,5.975,15.361,0.008,0.0
0.183,5.927,4.699,15.443,0.026,0.0
0.24,4.46,3.828,15.549,0.016,0.0
0.749,1.39,2.739,15.184,0.015,0.0
0.72,1.212,2.976,15.14,0.006,0.0
0.824,0.937,3.12,15.016,-0.089,0.0
0.127,0.463,3.018,13.706,0.036,0.0
0.584,3.254,1.629,14.581,0.053,0.0
0.565,0.66,1.621,14.753,0.085,0.0
0.853,3.41,0.45,13.939,0.012,0.0
0.836,4.713,0.337,14.072,0.007,0.0
0.968,2.708,0.249,14.346,-0.016,0.0
0.387,2.077,1.735,15.354,-0.039,0.0
0.398,2.395,2.933,15.532,0.059,0.0
0.327,2.428,1.813,15.518,0.036,0.0
0.635,1.999,0.061,15.292,-0.005,0.0
0.279,4.784,0.637,14.664,0.016,0.0
0.162,2.572,1.5,14.58,0.057,0.0
0.21,1.923,15.689,12.954,0.112,0.0
0.434,0.844,4.415,12.837,-0.246,0.0
0.271,1.323,5.378,12.849,0.17,0.0
0.416,2.122,5.392,13.508,0.098,0.0
0.409,2.225,4.438,13.477,0.018,0.0
0.262,3.185,5.914,13.392,0.128,0.0
0.23,4.172,5.796,14.956,0.218,0.0
0.085,11.333,15.115,15.14,0.246,0.0
0.134,6.66,9.398,15.292,0.101,0.0
0.759,2.52,2.074,13.84,0.084,0.0
0.77,1.92,1.572,14.162,0.031,0.0
0.675,2.107,1.621,13.957,0.036,0.0
0.601,2.387,1.541,13.442,-0.214,0.0
0.733,1.376,1.151,13.28,-0.11,0.0
0.822,1.461,1.286,13.478,-0.026,0.0
0.15,9.243,7.907,13.92,0.068,0.0
0.103,13.87,11.417,14.067,0.135,0.0
0.054,35.477,17.746,14.134,0.107,0.0
0.672,1.436,1.373,13.419,0.145,1.0
0.292,0.205,1.514,13.547,0.171,1.0
0.614,1.341,1.041,13.703,0.069,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,0.148,14.649,0.092,1.0
0.616,2.062,1.508,14.571,0.074,1.0
0.622,1.136,0.098,14.945,0.043,1.0
0.731,1.134,0.852,15.429,0.041,1.0
0.747,0.97,0.859,15.636,0.037,1.0
0.485,1.984,3.443,13.503,0.059,1.0
0.599,1.421,2.111,13.788,0.033,1.0
0.685,1.229,1.801,13.965,0.009,1.0
0.675,0.869,1.083,13.874,0.096,1.0
0.828,0.878,0.992,14.654,0.018,1.0
0.696,1.079,3.177,14.413,0.072,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.743,0.921,1.933,13.546,0.185,1.0
0.752,1.488,2.063,13.671,0.202,1.0
1.074,1.367,1.418,13.605,0.13,1.0
0.297,0.154,0.94,16.088,0.064,1.0
0.268,1.354,0.861,16.111,0.025,1.0
0.09,1.484,2.472,16.149,0.019,1.0
0.405,3.01,2.249,15.232,0.022,1.0
0.488,2.13,2.073,15.416,0.03,1.0
0.528,1.87,2.078,15.514,0.071,1.0
0.715,1.106,1.3,15.805,0.055,1.0
0.66,1.287,1.822,15.719,0.018,1.0
0.678,1.252,1.754,15.89,0.029,1.0
0.631,1.578,1.041,14.786,0.02,1.0
0.635,1.733,1.055,14.82,0.026,1.0
0.659,1.585,1.171,14.771,0.028,1.0
0.689,1.368,0.95,12.36,-0.033,1.0
0.464,1.238,1.069,12.662,-0.162,1.0
0.845,1.356,0.552,12.717,-0.352,1.0
0.548,1.741,0.697,16.077,0.002,1.0
0.595,1.234,0.758,16.058,0.0,1.0
0.768,0.919,0.355,16.536,-0.08,1.0
0.535,3.214,3.409,12.725,0.186,1.0
0.588,2.784,4.476,12.623,0.303,1.0
0.683,2.317,4.213,12.453,0.327,1.0
0.42,1.553,1.755,15.717,0.012,1.0
0.397,2.599,2.095,15.712,0.028,1.0
0.348,2.223,1.268,15.703,0.021,1.0
0.592,1.518,2.17,14.555,0.047,1.0
0.608,1.401,1.509,14.678,0.014,1.0
0.661,0.001,0.941,14.767,-0.013,1.0
1.48,0.225,0.001,12.488,-0.078,1.0
1.575,0.324,0.044,12.488,-0.009,1.0
0.638,0.87,1.662,15.163,-0.005,1.0
0.624,0.906,1.764,15.229,0.03,1.0
0.631,1.037,1.033,8.626,0.086,1.0
0.27,2.783,10.977,13.105,0.19,1.0
0.471,2.03,6.381,13.449,0.224,1.0
0.292,3.383,9.848,13.567,0.22,1.0
0.701,1.253,3.098,15.845,0.056,1.0
0.799,0.047,2.534,15.942,0.048,1.0
0.772,1.24,2.471,16.078,0.054,1.0
1.181,0.804,0.944,12.801,0.075,1.0
0.464,1.293,2.927,12.743,0.092,1.0
0.514,1.105,2.764,12.839,0.082,1.0
0.386,2.301,6.078,12.668,0.399,1.0
0.723,1.327,1.854,13.612,0.125,1.0
0.695,1.38,1.813,13.589,0.085,1.0
0.638,1.322,4.07,13.505,0.286,1.0
0.555,1.614,3.204,13.949,0.13,1.0
0.594,1.528,2.718,14.176,0.125,1.0
1.401,0.709,0.208,12.076,-0.146,1.0
0.916,0.616,1.27,12.629,0.316,1.0
0.75,0.489,2.46,12.725,0.29,1.0
0.445,9.164,0.0,11.167,-0.737,1.0
0.767,0.664,1.149,12.778,-0.779,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.775,1.025,3.294,13.079,0.112,1.0
0.731,0.932,4.179,12.889,0.126,1.0
0.842,0.673,1.911,13.274,-0.04,1.0
0.733,1.226,2.588,13.42,0.091,1.0
0.707,1.325,2.385,13.461,0.06,1.0
1.209,0.656,1.554,13.272,-0.565,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,1.479,14.649,0.092,1.0
0.616,2.062,2.277,14.571,0.074,1.0"""
impago = pd.read_csv(StringIO(IMPAGO_CSV))
print("Firms:", len(impago), "  Fraction in default:", round(impago["impago"].mean(), 3))
impago.describe().round(3).T[["mean", "min", "50%", "max"]]


In [ ]:

# The same partition in every notebook of the set: 146 firms for training, 73 for testing
RATIOS = ["deuda_activos", "razon_corriente", "ventas_deuda", "ln_activos", "roa"]
rng = np.random.default_rng(2)
orden = rng.permutation(len(impago))
es_prueba = np.zeros(len(impago), dtype=bool); es_prueba[orden[:73]] = True
impago["conjunto"] = np.where(es_prueba, "prueba", "entrenamiento")

X = impago[RATIOS]; y = impago["impago"]
X_train, y_train = X[~es_prueba], y[~es_prueba]
X_test, y_test = X[es_prueba], y[es_prueba]
print("Training:", len(X_train), "  Test:", len(X_test))
print("Fraction in default: training", round(y_train.mean(), 3), " test", round(y_test.mean(), 3))


## 2. Overfitting, measured

We train deeper and deeper trees and record the training and test accuracy.


In [ ]:

filas = []
for d in range(1, 11):
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    filas.append((d, m.get_n_leaves(), accuracy_score(y_train, m.predict(X_train)), accuracy_score(y_test, m.predict(X_test))))
curva = pd.DataFrame(filas, columns=["profundidad", "hojas", "entrenamiento", "prueba"])
fig = px.line(curva.melt(id_vars=["profundidad", "hojas"], var_name="conjunto", value_name="acierto"), x="profundidad", y="acierto", color="conjunto", markers=True,
              title="Accuracy as a function of tree depth")
fig.show()
curva.round(3)


**What to look at.** Training accuracy climbs without stopping until it reaches 100%. Test accuracy rises
up to depth 2 or 3 and then drops or stalls. Everything the tree learns after
that point is details of the 146 training firms that are of no use for the other 73.

Mind the scale: with 73 test firms, one percentage point is less than one firm.
Differences of 1 or 2 points are noise; what matters is the shape of the two curves.

This is what the tree looks like with no brake: 28 leaves for 146 firms.


In [ ]:

sin_freno = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
print(f"Leaves: {sin_freno.get_n_leaves()}   depth: {sin_freno.get_depth()}")
print(f"Training accuracy {accuracy_score(y_train, sin_freno.predict(X_train)):.3f}   test {accuracy_score(y_test, sin_freno.predict(X_test)):.3f}")
plt.figure(figsize=(22, 8))
plot_tree(sin_freno, feature_names=RATIOS, class_names=["pays", "defaults"], filled=True, fontsize=6)
plt.title("The tree with no brake: every leaf is a pure group, many with 1 or 2 firms")
plt.show()


## 3. Pre-pruning: not letting it grow

Stopping rules applied while building: maximum depth (`max_depth`), minimum firms
per leaf (`min_samples_leaf`), minimum to split a node (`min_samples_split`), minimum impurity
improvement (`min_impurity_decrease`). These are the hyperparameters of any library. The minimum
per leaf has a direct business reading: "I do not want rules based on fewer than 20
firms".


In [ ]:

filas = []
for msl in [1, 3, 5, 10, 20, 30]:
    m = DecisionTreeClassifier(min_samples_leaf=msl, random_state=0).fit(X_train, y_train)
    filas.append((msl, m.get_n_leaves(), accuracy_score(y_train, m.predict(X_train)), accuracy_score(y_test, m.predict(X_test))))
pd.DataFrame(filas, columns=["min per leaf", "leaves", "training", "test"]).round(3)


With 20 or 30 firms per leaf every leaf is a statistically meaningful group and the tree ends up
with 4 to 6 leaves. Test accuracy does not improve in a straight line as the minimum rises: again,
with 73 firms there is noise. The lesson is not "20 is the number"; it is that any tree of 3 to 8
leaves performs about the same and none with 20 or more beats them.

## 4. Post-pruning: cost-complexity

Breiman's alternative: let the full tree grow and then **cut back** the branches
that contribute little. For each internal node we compute how much error the branch hanging from
it saves and how many leaves it costs:

$$
\alpha_{ef}(\text{node}) = \frac{R(\text{node as a leaf}) - R(\text{branch})}{\text{leaves of the branch} - 1}
$$

where $R$ is the training error (fraction of misclassified firms, weighted by
how many pass through there). A small $\alpha_{ef}$ means "many leaves to save little error".
The branch with the smallest $\alpha_{ef}$ is pruned first, everything is recomputed, and so on: that gives a
**sequence** of smaller and smaller trees, indexed by $\alpha$. Choosing $\alpha$ is choosing the
size of the tree, and it is chosen with data the tree has not seen.


In [ ]:

camino = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(X_train, y_train)
alfas = camino.ccp_alphas[:-1]     # the last alpha leaves only the root
filas = []
for a in alfas:
    m = DecisionTreeClassifier(random_state=0, ccp_alpha=a).fit(X_train, y_train)
    filas.append((a, m.get_n_leaves(), accuracy_score(y_train, m.predict(X_train)), accuracy_score(y_test, m.predict(X_test))))
poda = pd.DataFrame(filas, columns=["alfa", "hojas", "entrenamiento", "prueba"])
fig = px.line(poda.melt(id_vars=["alfa", "hojas"], var_name="conjunto", value_name="acierto"), x="hojas", y="acierto", color="conjunto", markers=True,
              title="The pruning sequence: accuracy as a function of the remaining leaves")
fig.update_xaxes(autorange="reversed")
fig.show()
poda.round(4)


**How to read the table.** From top to bottom $\alpha$ rises and the tree loses leaves. Training
accuracy always drops (that is what pruning sacrifices). Test accuracy has no clean
trend: it oscillates between 73% and 81% from 28 leaves down to 3, and only falls at 2 leaves.
With 73 test firms, those oscillations are one to three firms. The correct reading
is that the 25 extra leaves contribute nothing, not that there is a magic $\alpha$.

## 5. Choosing the complexity without looking at the test set: cross-validation

So far we chose the depth by looking at test accuracy. That is cheating: the 73
test firms are there to give the final grade, not to decide the model. If we use them to
choose, the grade is inflated.

The solution is **cross-validation**: split the training data into $k$ folds (here 5),
train on 4 and measure on the fifth, rotate, and average. Every training firm serves
once for measuring, without touching the test set. With that we choose the depth and the $\alpha$; the
test set is used only once, at the end.


In [ ]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
filas = []
for d in range(1, 9):
    m = DecisionTreeClassifier(max_depth=d, random_state=0)
    notas = cross_val_score(m, X_train, y_train, cv=cv, scoring="accuracy")
    filas.append((d, notas.mean(), notas.std()))
cv_prof = pd.DataFrame(filas, columns=["profundidad", "acierto validación cruzada", "desviación entre pedazos"])
mejor_d = int(cv_prof.loc[cv_prof["acierto validación cruzada"].idxmax(), "profundidad"])
print("Depth chosen by cross-validation:", mejor_d)
cv_prof.round(3)


In [ ]:

busqueda = GridSearchCV(DecisionTreeClassifier(random_state=0),
                        {"ccp_alpha": alfas, "min_samples_leaf": [1, 5, 10, 20]},
                        cv=cv, scoring="accuracy").fit(X_train, y_train)
print("Best parameters:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in busqueda.best_params_.items()})
print(f"Cross-validation accuracy: {busqueda.best_score_:.3f}")
final = busqueda.best_estimator_
print(f"Leaves of the chosen tree: {final.get_n_leaves()}")
print(f"Test accuracy (only once, at the end): {accuracy_score(y_test, final.predict(X_test)):.3f}")
print()
print(export_text(final, feature_names=RATIOS))


**What happened, and an uncomfortable lesson.** Cross-validation over depth chose a tree
with a single split, and the combined search chose a different one; on test data, the chosen one does worse
(75%) than the depth-2 tree we had picked by looking at the test set (82%). It is not that
cross-validation is wrong: look at the column of standard deviation across folds, 4 to 10 points,
larger than the differences between depths. With 146 firms there is not enough information
to tell a 2-leaf tree from a 7-leaf one, and the earlier 82% was partly
luck with those 73 firms. What cross-validation guarantees is not the best grade, but
an **honest** grade; the best grade comes from more data.

## 6. What to take away

- A tree with no brake memorizes: 100% on training says nothing. The only curve that matters
  is the one on data it has not seen.
- Depth, minimum per leaf and $\alpha$ are three knobs for the same thing: how much complexity it is
  allowed. All of them are chosen with cross-validation, never with training accuracy and
  never with the test set.
- With little data (146 firms) any tree of 2 to 8 leaves performs about the same and cross-validation
  cannot tell them apart; there is no information for more, and the rest is well-memorized
  noise. A test accuracy chosen by looking at the test set is inflated.
- What pruning does **not** fix is instability: change a few rows and the tree changes. That is
  what ensembles take care of, in Notebook 4.

## Exercises

**Exercise 1.** Repeat the curve from section 2 with another partition (`default_rng(5)` instead of
`default_rng(2)`). Does the depth at which test accuracy peaks change? Does the root
of the depth-2 tree change? That is instability.

**Exercise 2.** Use `min_impurity_decrease` as the stopping rule (try 0.005, 0.01, 0.02,
0.05). How many leaves remain with each value and how does it do on test data? It is the "minimum
gain" version of pre-pruning.

**Exercise 3.** Cross-validation with 5 folds trains on 117 firms and measures on 29.
Repeat the depth selection with 10 folds and with 3. Does the chosen depth change? What
happens to the standard deviation across folds?


## Solutions


In [ ]:

# SOLUTION 1 -- another partition
rng5 = np.random.default_rng(5); orden5 = rng5.permutation(len(impago))
prueba5 = np.zeros(len(impago), dtype=bool); prueba5[orden5[:73]] = True
Xtr5, ytr5, Xte5, yte5 = X[~prueba5], y[~prueba5], X[prueba5], y[prueba5]
for d in range(1, 7):
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr5, ytr5)
    print(f"depth {d}: training {accuracy_score(ytr5, m.predict(Xtr5)):.3f}   test {accuracy_score(yte5, m.predict(Xte5)):.3f}")
m2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(Xtr5, ytr5)
print("\nRoot with partition 5:", RATIOS[m2.tree_.feature[0]], "≤", round(float(m2.tree_.threshold[0]), 3), "  (with partition 2 it was ventas_deuda ≤ 1.278)")


In [ ]:

# SOLUTION 2 -- minimum gain
for mid in [0.005, 0.01, 0.02, 0.05]:
    m = DecisionTreeClassifier(min_impurity_decrease=mid, random_state=0).fit(X_train, y_train)
    print(f"min_impurity_decrease {mid:.3f}: {m.get_n_leaves():2d} leaves   training {accuracy_score(y_train, m.predict(X_train)):.3f}   test {accuracy_score(y_test, m.predict(X_test)):.3f}")


In [ ]:

# SOLUTION 3 -- how many folds
for k in [3, 5, 10]:
    cvk = StratifiedKFold(n_splits=k, shuffle=True, random_state=0)
    res = [(d, cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0), X_train, y_train, cv=cvk).mean(),
            cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0), X_train, y_train, cv=cvk).std()) for d in range(1, 9)]
    mejor = max(res, key=lambda t: t[1])
    print(f"{k:2d} folds: chosen depth {mejor[0]}, accuracy {mejor[1]:.3f}, std across folds {mejor[2]:.3f}")
print("With more folds each one measures on fewer firms: the standard deviation across folds grows, even though the average uses more data for training.")
